In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# EDA

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("=== DATA SHAPE & COLUMNS ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")

print("=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== SAMPLE HEAD ===")
print(df.head(2))

if 'answer' in df.columns:
    print("\n=== TARGET DISTRIBUTION ===")
    print(df['answer'].value_counts(dropna=False, normalize=True) * 100)

In [ ]:
import matplotlib.pyplot as plt

option_cols = ['A', 'B', 'C', 'D', 'E']
prompt_col = 'prompt'
answer_col = 'answer'


if prompt_col:
    df['prompt_word_len'] = df[prompt_col].astype(str).apply(lambda x: len(x.split()))
if answer_col:
    df['answer_word_len'] = df[answer_col].astype(str).apply(lambda x: len(x.split()))

for col in option_cols:
    df[f'{col}_word_len'] = df[col].astype(str).apply(lambda x: len(x.split()))

df['total_word_len'] = df[[c for c in df.columns if 'word_len' in c]].sum(axis=1)

print("=== LENGTH STATS (WORD COUNTS) ===")
print(df[[c for c in df.columns if 'word_len' in c]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
for col in option_cols:
    df[f'{col}_char_len'] = df[col].astype(str).apply(len)

if 'answer' in df.columns:
    def get_correct_option_len(row):
        ans = str(row['answer']).strip()
        for col in option_cols:
            if ans == col:
                return len(str(row[col]))
        return None

    df['correct_option_char_len'] = df.apply(get_correct_option_len, axis=1)
    
    print("=== OPTION LENGTH COMPARISON ===")
    print(f"Mean Correct Option Length: {df['correct_option_char_len'].mean():.2f} chars")
    print("Mean Option Lengths by Column:")
    for col in option_cols:
        print(f"  {col}: {df[f'{col}_char_len'].mean():.2f} chars")

In [ ]:
if prompt_col:
    exact_dup_prompts = df[prompt_col].duplicated().sum()
    print("=== DUPLICATE ANALYSIS ===")
    print(f"Exact Duplicate Prompts: {exact_dup_prompts} ({(exact_dup_prompts/len(df))*100:.2f}%)")

def check_duplicate_options(row):
    opts = [str(row[col]).strip() for col in option_cols]
    return len(opts) != len(set(opts))

df['has_duplicate_options'] = df.apply(check_duplicate_options, axis=1)
print(f"Questions with duplicate options: {df['has_duplicate_options'].sum()}")

In [ ]:
import re

def detect_patterns(text):
    text = str(text)
    has_latex = bool(re.search(r'\$|\\frac|\\sqrt|\\alpha|\\beta', text))
    has_code = bool(re.search(r'def\s+|class\s+|import\s+|for\s+.*in|#include', text))
    has_math_symbols = bool(re.search(r'[=+\-*/^<>≤≥≠±]', text))
    return pd.Series([has_latex, has_code, has_math_symbols], index=['has_latex', 'has_code', 'has_math_symbols'])

if prompt_col:
    pattern_df = df[prompt_col].apply(detect_patterns)
    print("=== DOMAIN & SYNTAX PATTERNS ===")
    print(pattern_df.mean() * 100)

In [ ]:
import transformers
from huggingface_hub import HfApi

print(f"=== TRANSFORMERS VERSION ===")
print(f"Version: {transformers.__version__}\n")

candidates = [
    "roberta-base",
    "roberta-large",
    "microsoft/deberta-v3-base",
    "microsoft/deberta-v3-small",
    "bert-base-uncased",
    "google/electra-base-discriminator"
]

print("=== CHECKING HF MODEL ACCESSIBILITY ===")
api = HfApi()

for model_id in candidates:
    try:
        info = api.model_info(model_id)
        print(f"✅ FOUND: '{model_id}'")
    except Exception as e:
        print(f"❌ NOT FOUND / ERROR: '{model_id}' -> {e}")

In [ ]:
import os
import torch
import wandb
from dataclasses import dataclass
from typing import Optional, Union
from transformers import (
AutoTokenizer, 
AutoModelForMultipleChoice, 
TrainingArguments, 
Trainer,
PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

# 1. automatic W&B authentication
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="roberta-base-baseline",
    reinit="finish_previous"
)

MODEL_NAME = "roberta-base"
MAX_LEN = 256
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
def preprocess_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

In [ ]:
gkf = GroupKFold(n_splits=5)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df = df[df['fold'] == 0].reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(preprocess_function, batched=True)

In [ ]:
def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

In [ ]:
#previous underperforming parameters:
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     warmup_steps=50,                  # Updated from warmup_ratio to avoid deprecation warning
#     max_grad_norm=1.0,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=2,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_steps=20,
#     report_to="wandb",
#     load_best_model_at_end=True,
#     metric_for_best_model="accuracy",
#     fp16=torch.cuda.is_available()
# )

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
training_args = TrainingArguments(
        output_dir=f"./results_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        save_only_model=True,
        learning_rate=1.5e-5,             # 1. Lowered LR for fine-grained convergence
        warmup_steps=100,                 # Smooth warmup over the first ~10% of total steps
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,    # Effective batch size = 8
        num_train_epochs=4,               # 2. Increased to 4 epochs
        label_smoothing_factor=0.05,      # 3. Prevents overconfidence, boosts MAP@3 ranking
        weight_decay=0.01,
        logging_steps=20,
        report_to="wandb",
        fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
wandb.finish()

In [ ]:
from torch.utils.data import DataLoader

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(f"Test Set Shape: {test_df.shape}")
print(f"Sample Submission Columns: {list(sample_sub.columns)}")

def preprocess_test_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

test_dataset = Dataset.from_pandas(test_df).map(preprocess_test_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

class TestDataCollatorForMultipleChoice:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_input_cols} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=True,
            return_tensors="pt",
        )

        return {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=TestDataCollatorForMultipleChoice(tokenizer=tokenizer)
)

all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)  # Shape: (500, 5)

id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]

top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

print("\n=== TOP PREDICTION PATTERNS ===")
print(submission_df[target_col_name].value_counts().head(10))

# Ensemble :

In [ ]:
oof_predictions = np.zeros((len(df), 5))

for fold in range(NUM_FOLDS):
    print(f"\n========================== TRAINING FOLD {fold + 1}/{NUM_FOLDS} ==========================")
    
    wandb.init(
        project="23f2002267-t22026", 
        name=f"roberta-base-fold{fold}",
        reinit="finish_previous"
    )

    fold_train = df[df['fold'] != fold].reset_index(drop=True)
    fold_val = df[df['fold'] == fold].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(fold_train).map(preprocess_function, batched=True)
    val_dataset = Dataset.from_pandas(fold_val).map(preprocess_function, batched=True)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    training_args = TrainingArguments(
        output_dir=f"./results_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",               
        save_only_model=True,             
        learning_rate=2e-5,
        warmup_steps=50,
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=20,
        report_to="wandb",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer, 
        data_collator=TrainDataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    # Save lightweight final model for this fold
    model.save_pretrained(f"./model_fold_{fold}")
    tokenizer.save_pretrained(f"./model_fold_{fold}")
    
    # Clean GPU memory between folds
    del model, trainer
    torch.cuda.empty_cache()
    wandb.finish()

print("\n========================== STARTING 5-FOLD ENSEMBLE INFERENCE ==========================")

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

def preprocess_test_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

test_dataset = Dataset.from_pandas(test_df).map(preprocess_test_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

class TestDataCollatorForMultipleChoice:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_input_cols} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=True,
            return_tensors="pt",
        )

        return {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=TestDataCollatorForMultipleChoice(tokenizer=tokenizer)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_fold_logits = []


for fold in range(NUM_FOLDS):
    print(f"Running inference with Fold {fold} model...")
    fold_model = AutoModelForMultipleChoice.from_pretrained(f"./model_fold_{fold}")
    fold_model.eval()
    fold_model.to(device)

    fold_logits = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
            fold_logits.append(outputs.logits.cpu().numpy())

    all_fold_logits.append(np.concatenate(fold_logits, axis=0))
    del fold_model
    torch.cuda.empty_cache()


mean_logits = np.mean(all_fold_logits, axis=0)  # Shape: (500, 5)


top3_indices = np.argsort(mean_logits, axis=1)[:, ::-1][:, :3]
top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]


id_col_name = sample_sub.columns[0]
target_col_name = sample_sub.columns[1]

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== ENSEMBLED MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))